In [19]:
import pandas as pd
import os

In [20]:
SAMPLE_DIR = r"D:\projects\Healthcare\data\raw_sample"
df = pd.read_csv(os.path.join(SAMPLE_DIR, "stroke_sliced.csv"))

In [21]:
#Missing value detection

missing = df.isna().sum()
missing_pct = (df.isna().mean()*100).round(2)

In [22]:
#only show columns that actually have missing values
missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
})
missing_report = missing_report[missing_report["missing_count"] > 0]

print("columns with missing values:\n")
print(missing_report)
print(f"\nTotal missing cells: {df.isna().sum().sum()}")

columns with missing values:

          missing_count  missing_pct
Symptoms           2500        16.67

Total missing cells: 2500


In [23]:
def treat_missing(df, drop_threshold=0.5):
    """Columns with missing fraction above drop_threshold are dropped."""
    df = df.copy()
    report = {"dropped_columns": [], "filled_columns":{}}

    for col in df.columns:
        miss_frac = df[col].isna().mean()

        if miss_frac == 0:
            continue

        if miss_frac > drop_threshold:
            df = df.drop(columns=[col])
            report["dropped_columns"].append(col)
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            fill_value = df[col].median()
            method = "median"
        else:
            fill_value = df[col].mode()[0]
            method = "mode"
        df[col] = df[col].fillna(fill_value)
        report["filled_columns"][col] = {
                "method": method,
                "fill_value": fill_value,
                "pct_filled": round(miss_frac *100,2),
        }
        return df, report
    
df_clean, missing_report = treat_missing(df)

print("Dropped columns:", missing_report["dropped_columns"])
print("\nFilled columns:")
for col, info in missing_report["filled_columns"].items():
     print(f"   {col:20} method={info['method']:8} value={info['fill_value']} ({info['pct_filled']}%filled)")
print(f"\nMissing after treatment: {df_clean.isna().sum().sum()}")






Dropped columns: []

Filled columns:
   Symptoms             method=mode     value=Difficulty Speaking (16.67%filled)

Missing after treatment: 0


Duplicate detection and removal

In [24]:
def remove_duplicates(df):

    df = df.copy()

    n_before = df.shape[0]
    n_duplicates = int(df.duplicated().sum())

    df = df.drop_duplicates()
    df = df.reset_index(drop=True)

    n_after = df.shape[0]

    report = {
        "rows_before": n_before,
        "duplicates_found": n_duplicates,
        "rows_after": n_after,
        "rows_removed": n_before - n_after,
    }
    return df, report
#Test
df_nodup, dup_report = remove_duplicates(df_clean)

print("Duplicate removal report:")
for key, value in dup_report.items():
    print(f" {key:<20}: {value}")

Duplicate removal report:
 rows_before         : 15000
 duplicates_found    : 0
 rows_after          : 15000
 rows_removed        : 0


Outlier detection and capping

In [25]:
def treat_outliers(df, factor=1.5, min_unique=15):
   
    df = df.copy()
    report = {}

    numeric_cols = df.select_dtypes(include="number").columns

    for col in numeric_cols:
        # Skip binary / low-cardinality columns (likely categorical)
        if df[col].nunique() < min_unique:
            continue

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - factor * IQR
        upper = Q3 + factor * IQR

        n_outliers = int(((df[col] < lower) | (df[col] > upper)).sum())
        if n_outliers > 0:
            df[col] = df[col].clip(lower=lower, upper=upper)
            report[col] = {
                "outliers": n_outliers,
                "lower_bound": round(lower, 2),
                "upper_bound": round(upper, 2),
            }

    return df, report


# Re-test
df_test = pd.read_csv(os.path.join(SAMPLE_DIR, "stroke_sliced.csv"))
df_out, out_report = treat_outliers(df_test)

print("Outlier columns treated:", len(out_report))
for col, info in out_report.items():
    print(f"  {col:<28} {info['outliers']} capped, bounds=[{info['lower_bound']}, {info['upper_bound']}]")

Outlier columns treated: 0


Master Cleaning Pipeline and Report

In [26]:
def clean_dataset(df, drop_threshold=0.5, outlier_factor=1.5):
    
    report = {"original_shape": df.shape}

    # Step 1: missing values
    df, missing_rep = treat_missing(df, drop_threshold=drop_threshold)
    report["missing"] = missing_rep

    # Step 2: duplicates
    df, dup_rep = remove_duplicates(df)
    report["duplicates"] = dup_rep

    # Step 3: outliers
    df, outlier_rep = treat_outliers(df, factor=outlier_factor)
    report["outliers"] = outlier_rep

    report["final_shape"] = df.shape
    return df, report


def print_cleaning_report(report):
    
    print("=" * 55)
    print("  CLEANING REPORT")
    print("=" * 55)
    print(f"Original shape : {report['original_shape']}")
    print(f"Final shape    : {report['final_shape']}")

    # Missing
    m = report["missing"]
    print(f"\n[Missing Values]")
    print(f"  Dropped columns : {m['dropped_columns'] or 'None'}")
    print(f"  Filled columns  : {len(m['filled_columns'])}")
    for col, info in m["filled_columns"].items():
        print(f"     - {col} ({info['method']}, {info['pct_filled']}%)")

    # Duplicates
    d = report["duplicates"]
    print(f"\n[Duplicates]")
    print(f"  Removed : {d['rows_removed']} rows")

    # Outliers
    o = report["outliers"]
    print(f"\n[Outliers] (capped in {len(o)} columns)")
    for col, info in o.items():
        print(f"     - {col}: {info['outliers']} capped")


# Run full pipeline on fresh stroke data
df_raw = pd.read_csv(os.path.join(SAMPLE_DIR, "stroke_sliced.csv"))
df_cleaned, full_report = clean_dataset(df_raw)
print_cleaning_report(full_report)

  CLEANING REPORT
Original shape : (15000, 22)
Final shape    : (15000, 22)

[Missing Values]
  Dropped columns : None
  Filled columns  : 1
     - Symptoms (mode, 16.67%)

[Duplicates]
  Removed : 0 rows

[Outliers] (capped in 0 columns)


In [27]:
df_cleaned.head()


,Patient ID,Patient Name,Age,Gender,Hypertension,Heart Disease,Marital Status,Work Type,Residence Type,Average Glucose Level,...,Alcohol Intake,Physical Activity,Stroke History,Family History of Stroke,Dietary Habits,Stress Levels,Blood Pressure Levels,Cholesterol Levels,Symptoms,Diagnosis
0,18153,Mamooty Khurana,56,Male,0,1,Married,Self-employed,Rural,130.91,...,Social Drinker,Moderate,0,Yes,Vegan,3.48,140/108,"HDL: 68, LDL: 133","Difficulty Speaking, Headache",Stroke
1,62749,Kaira Subramaniam,80,Male,0,0,Single,Self-employed,Urban,183.73,...,Never,Low,0,No,Paleo,1.73,146/91,"HDL: 63, LDL: 70","Loss of Balance, Headache, Dizziness, Confusion",Stroke
2,32145,Dhanush Balan,26,Male,1,1,Married,Never Worked,Rural,189.00,...,Rarely,High,0,Yes,Paleo,7.31,154/97,"HDL: 59, LDL: 95","Seizures, Dizziness",Stroke
3,6154,Ivana Baral,73,Male,0,0,Married,Never Worked,Urban,185.29,...,Frequent Drinker,Moderate,0,No,Paleo,5.35,174/81,"HDL: 70, LDL: 137","Seizures, Blurred Vision, Severe Fatigue, Head...",No Stroke
4,48973,Darshit Jayaraman,51,Male,1,1,Divorced,Self-employed,Urban,177.34,...,Rarely,Low,0,Yes,Pescatarian,6.84,121/95,"HDL: 65, LDL: 68",Difficulty Speaking,Stroke


In [28]:
df.isnull().sum()

Patient ID                     0
Patient Name                   0
Age                            0
Gender                         0
Hypertension                   0
Heart Disease                  0
Marital Status                 0
Work Type                      0
Residence Type                 0
Average Glucose Level          0
Body Mass Index (BMI)          0
Smoking Status                 0
Alcohol Intake                 0
Physical Activity              0
Stroke History                 0
Family History of Stroke       0
Dietary Habits                 0
Stress Levels                  0
Blood Pressure Levels          0
Cholesterol Levels             0
Symptoms                    2500
Diagnosis                      0
dtype: int64

In [29]:
df_cleaned.isnull().sum()

Patient ID                  0
Patient Name                0
Age                         0
Gender                      0
Hypertension                0
Heart Disease               0
Marital Status              0
Work Type                   0
Residence Type              0
Average Glucose Level       0
Body Mass Index (BMI)       0
Smoking Status              0
Alcohol Intake              0
Physical Activity           0
Stroke History              0
Family History of Stroke    0
Dietary Habits              0
Stress Levels               0
Blood Pressure Levels       0
Cholesterol Levels          0
Symptoms                    0
Diagnosis                   0
dtype: int64

In [30]:
import sys
import os

# Re-add src to path (safe to run again)
src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.append(src_path)

from data_cleaning import clean_dataset

SAMPLE_DIR = r"D:\projects\Healthcare\data\raw_sample"
df_raw = pd.read_csv(os.path.join(SAMPLE_DIR, "stroke_sliced.csv"))
df_cleaned, report = clean_dataset(df_raw)

print("Original:", report["original_shape"])
print("Final   :", report["final_shape"])
print("Filled  :", list(report["missing"]["filled_columns"].keys()))
print("Dup removed:", report["duplicates"]["rows_removed"])
print("Outlier cols:", len(report["outliers"]))

2026-06-30 17:13:04,904 | data_cleaning | INFO | Cleaning started. Input shape: (15000, 22)
2026-06-30 17:13:04,942 | data_cleaning | INFO | Filled 'Symptoms' with mode (16.7% missing)
2026-06-30 17:13:05,030 | data_cleaning | INFO | Removed 0 duplicate rows
2026-06-30 17:13:05,047 | data_cleaning | INFO | Skipped 'Hypertension' for outliers (2 unique values)
2026-06-30 17:13:05,047 | data_cleaning | INFO | Skipped 'Heart Disease' for outliers (2 unique values)
2026-06-30 17:13:05,057 | data_cleaning | INFO | Skipped 'Stroke History' for outliers (2 unique values)
2026-06-30 17:13:05,067 | data_cleaning | INFO | Cleaning finished. Output shape: (15000, 22)


Original: (15000, 22)
Final   : (15000, 22)
Filled  : ['Symptoms']
Dup removed: 0
Outlier cols: 0


In [31]:
df_check = pd.read_csv(os.path.join(SAMPLE_DIR, "stroke_sliced.csv"))
numeric_cols = df_check.select_dtypes(include="number").columns

print("Numeric columns + unique counts:\n")
for col in numeric_cols:
    print(f"  {col:<28} unique={df_check[col].nunique()}")

Numeric columns + unique counts:

  Patient ID                   unique=15000
  Age                          unique=73
  Hypertension                 unique=2
  Heart Disease                unique=2
  Average Glucose Level        unique=9215
  Body Mass Index (BMI)        unique=2490
  Stroke History               unique=2
  Stress Levels                unique=1001
